In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

In [ ]:
df=pd.read_csv('diabetes.csv')

In [ ]:
column=['Glucose','BloodPressure','BMI']
df[column]=df[column].replace(0,np.nan)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Outcome']),df['Outcome'],random_state=42,test_size=0.2)

In [ ]:
x_train.sample()

In [ ]:
ct1=ColumnTransformer([
    ('Impute_glucose',SimpleImputer(strategy='median'),[1]),
    ('Impute_BMI',SimpleImputer(strategy='median'),[5]),
    ('Impute_bloodpressure',SimpleImputer(),[2]),
],remainder='passthrough')

In [ ]:
ct2=ColumnTransformer([
    ('scaling',RobustScaler(),[0,1,2,3,4,5,6,7])
],remainder='passthrough')

In [ ]:
pipe=Pipeline([
    ('ct1',ct1),
    ('ct2',ct2),
])

In [ ]:
x_train_transform=pipe.fit_transform(x_train)

In [ ]:
x_test_transform=pipe.fit_transform(x_test)

### Logistics Regression model

In [ ]:
def gd(X,y):
    
    X = np.insert(X,0,1,axis=1)
    weights = np.ones(X.shape[1])
    lr = 0.5
    
    for i in range(5000):
        y_hat = sigmoid(np.dot(X,weights))
        weights = weights + lr*(np.dot((y-y_hat),X)/X.shape[0])
        
    return weights[1:],weights[0]
        

In [ ]:
def sigmoid(z):
    return 1/(1 + np.exp(-z))

In [ ]:
def predict(x):
    y= np.where(
            x>0,
            1,
            0
        )
    return y

#### Training model

In [ ]:
w,w0=gd(x_train_transform,y_train)

#### Testing model

In [ ]:
y_output=np.dot(x_test_transform,w)+w0

In [ ]:
y_pred=predict(y_output)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
accuracy

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))